In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/SIMUTEH/simuteh-2330511078")
SRC = PROJECT / "src"

SRC.mkdir(parents=True, exist_ok=True)

print("Folder kerja:", SRC)
print("Folder tersedia:", SRC.is_dir())

Mounted at /content/drive
Folder kerja: /content/drive/MyDrive/SIMUTEH/simuteh-2330511078/src
Folder tersedia: True


In [2]:
%%writefile /content/drive/MyDrive/SIMUTEH/simuteh-2330511078/src/utils_tepi.py

"""utils_tepi.py: fungsi bantu evaluasi deteksi tepi dan morfologi (P5-P6 SIMUTEH)."""

import cv2
import numpy as np


def tepi_acuan(mask):
    """Tepi acuan selebar 1 piksel dari sebuah mask biner (255 = objek)."""
    m = (mask > 0).astype(np.uint8)
    return ((m - cv2.erode(m, np.ones((3, 3), np.uint8))) > 0).astype(np.uint8) * 255


def skor_tepi(prediksi, acuan, toleransi=2):
    """Presisi, recall, dan F1 tepi dengan toleransi jarak (piksel).

    Presisi: porsi piksel tepi prediksi yang berjarak <= toleransi dari tepi acuan.
    Recall : porsi piksel tepi acuan yang berjarak <= toleransi dari tepi prediksi.
    """
    p = prediksi > 0
    a = acuan > 0

    if p.sum() == 0 or a.sum() == 0:
        return 0.0, 0.0, 0.0

    jarak_ke_a = cv2.distanceTransform((~a).astype(np.uint8), cv2.DIST_L2, 3)
    jarak_ke_p = cv2.distanceTransform((~p).astype(np.uint8), cv2.DIST_L2, 3)

    presisi = float((jarak_ke_a[p] <= toleransi).mean())
    recall = float((jarak_ke_p[a] <= toleransi).mean())

    f1 = 0.0 if presisi + recall == 0 else 2 * presisi * recall / (presisi + recall)

    return round(presisi, 3), round(recall, 3), round(f1, 3)


def iou(a, b):
    """Intersection over Union dua mask biner."""
    a, b = a > 0, b > 0
    gabungan = (a | b).sum()

    return 1.0 if gabungan == 0 else float((a & b).sum() / gabungan)


def mask_daun_awal(img_bgr):
    """Mask daun sementara: ambang Otsu pada kanal saturasi (dibahas tuntas di P7)."""
    s = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)[:, :, 1]
    _, m = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return m


def bercak_awal(img_bgr, mask_daun, kanal=2, k=4.0):
    """Citra biner bercak sementara: piksel daun pada kanal terpilih yang lebih
    terang dari median daun + k x MAD (simpangan median absolut, dikali 1,4826).

    Median dan MAD dipakai karena tidak ikut terseret oleh bercak yang luas.
    Hasilnya sengaja masih kasar: memuat titik derau, potongan tulang daun,
    dan tepi daun yang harus dibersihkan dengan operasi morfologi (P6).
    """
    c = img_bgr[:, :, kanal].astype(np.float32)
    m = mask_daun > 0

    med = np.median(c[m])
    mad = 1.4826 * np.median(np.abs(c[m] - med))

    return (((c > med + k * mad) & m) * 255).astype(np.uint8)

Writing /content/drive/MyDrive/SIMUTEH/simuteh-2330511078/src/utils_tepi.py


In [3]:
import sys
import importlib
from pathlib import Path

SRC = Path("/content/drive/MyDrive/SIMUTEH/simuteh-2330511078/src")

file = SRC / "utils_tepi.py"

print("File tersedia:", file.is_file())
print("Ukuran file:", file.stat().st_size if file.is_file() else "Tidak ditemukan")
print("Isi folder src:", sorted(p.name for p in SRC.iterdir()))

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

importlib.invalidate_caches()
import utils_tepi

print("Import berhasil!")
print("Lokasi modul:", utils_tepi.__file__)

File tersedia: True
Ukuran file: 2271
Isi folder src: ['Untitled0.ipynb', '__pycache__', 'buat_dataset_sintesis.py', 'praproses.py', 'praproses_acuan.py', 'utils_cv.py', 'utils_tepi.py']
Import berhasil!
Lokasi modul: /content/drive/MyDrive/SIMUTEH/simuteh-2330511078/src/utils_tepi.py
